# Практическая работа №3. Исследование графовых структур в наборе финансовых транзакций

**ФИО:** Бухарин М. А.

**Группа:** 2384

**Ссылка на репозиторий:** https://github.com/bulterier94/vd_lab_2/tree/main/lab3

Цель работы — исследовать графовую структуру транзакций и выделить устойчивые паттерны мошеннического поведения на основе визуализации.

## 1. Постановка задачи

В наборе данных содержатся журналы транзакций в системе мобильных денежных переводов.  
По условию нужно:

- загрузить и изучить набор данных;
- построить графовые визуализации;
- выделить **2 мошеннические схемы** и подтвердить выводы иллюстрациями.

Внутри данных есть три мошеннических метки:

- `F_bot` — подозрительная активность в переводах между пользователями;
- `F-Mule-With` и `F_SevWith` — два варианта вывода средств через агентскую сеть.

Для итогового отчёта ниже я рассматриваю **две схемы**:

1. **Botnet-схема** — `F_bot`;
2. **Cash-out / вывод средств через агентов** — объединение `F-Mule-With` и `F_SevWith` как один семейный паттерн.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

# Путь к CSV в текущей папке
DATA_CSV = Path('FinFraud_Labelled.csv')

# Чтение файла
df = pd.read_csv(DATA_CSV, sep='|', header=None)

# Убираем пустую техническую строку в начале файла
df = df.iloc[1:].reset_index(drop=True)

# Переименование колонок
columns = {
    0:  'label',
    1:  'sender_user',
    2:  'receiver_user',
    3:  'sender_account',
    4:  'receiver_account',
    5:  'amount',
    6:  'txn_type',
    7:  'status',
    8:  'sender_balance_before',
    9:  'sender_balance_after',
    10: 'receiver_balance_before',
    11: 'receiver_balance_after',
    12: 'flag_1',
    13: 'flag_2',
    14: 'flag_3',
    15: 'flag_4',
    16: 'sender_time',
    17: 'receiver_time',
    18: 'sender_account_repeat',
    19: 'service_1',
    20: 'service_2',
    21: 'transaction_id',
    22: 'txn_time',
    23: 'sender_type',
    24: 'receiver_type',
}
df = df.rename(columns=columns)

# Типы данных
num_cols = [
    'amount',
    'sender_balance_before',
    'sender_balance_after',
    'receiver_balance_before',
    'receiver_balance_after'
]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

for c in ['sender_time', 'receiver_time', 'txn_time']:
    df[c] = pd.to_datetime(df[c], format='%d/%m/%Y %H:%M:%S', errors='coerce')

# Вывод
display(df.head())
print('Размер таблицы:', df.shape)
print('Метки транзакций:', df['label'].value_counts().to_dict())

C:\Users\bulterier04\AppData\Local\Temp\ipykernel_6784\1565996372.py:14: DtypeWarning: Columns (0: 12, 1: 13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_CSV, sep='|', header=None)


,label,sender_user,receiver_user,sender_account,receiver_account,amount,txn_type,status,sender_balance_before,sender_balance_after,...,flag_4,sender_time,receiver_time,sender_account_repeat,service_1,service_2,transaction_id,txn_time,sender_type,receiver_type
0,N-RegC2C,PN_EU_3_4,PN_EU_0_883,EUAcc3_4,EUAcc0_883,68897.74,Ind,SU,1.000000e+08,9.993041e+07,...,NaN,2011-06-01 00:09:01,2011-06-01 00:09:01,EUAcc3_4,NaN,NaN,C2C201161.099,2011-06-01 00:09:01,EU,EU
1,N-RegC2C,PN_EU_1_139,PN_EU_0_754,EUAcc1_139,EUAcc0_754,68945.47,Ind,SU,1.000000e+08,9.993037e+07,...,NaN,2011-06-01 00:15:23,2011-06-01 00:15:23,EUAcc1_139,NaN,NaN,C2C201161.01515,2011-06-01 00:15:23,EU,EU
2,N-RegDep,PN_Ret2,PN_EU_3_17,RAcc2,EUAcc3_17,9715.41,Dt,SU,1.000000e+09,9.999903e+08,...,NaN,2011-06-01 00:22:07,2011-06-01 00:22:07,RAcc2,NaN,NaN,Dp201161.02222,2011-06-01 00:22:07,RET,EU
3,N-RegDep,PN_Ret1,PN_EU_0_266,RAcc1,EUAcc0_266,79303.74,Dt,SU,1.000000e+09,9.999207e+08,...,NaN,2011-06-01 00:22:35,2011-06-01 00:22:35,RAcc1,NaN,NaN,Dp201161.02222,2011-06-01 00:22:35,RET,EU
4,N_Reg_RC,PN_EU_0_905,operator,EUAcc0_905,A0,929.92,ArRC,SU,1.000000e+08,9.999907e+07,...,NaN,2011-06-01 00:29:56,2011-06-01 00:29:56,EUAcc0_905,NaN,NaN,Rc201161.02929,2011-06-01 00:29:56,EU,operator


Размер таблицы: (54848, 25)
Метки транзакций: {'N_Reg_RC': 28312, 'N-RegDep': 12867, 'N-RegC2C': 7484, 'N_RegWith': 4064, 'F_bot': 721, 'F-Mule-With': 717, 'N_Reg_Merch': 443, 'F_SevWith': 240}


In [4]:
# Базовая сводка по мошенническим меткам
fraud = df[df['label'].str.startswith('F', na=False)].copy()

summary = []
for label in ['F_bot', 'F-Mule-With', 'F_SevWith']:
    sub = fraud[fraud['label'] == label]
    summary.append({
        'label': label,
        'rows': len(sub),
        'unique_senders': sub['sender_user'].nunique(),
        'unique_receivers': sub['receiver_user'].nunique(),
        'unique_edges': sub[['sender_user','receiver_user']].drop_duplicates().shape[0],
        'avg_tx_per_sender': len(sub) / sub['sender_user'].nunique(),
        'avg_tx_per_edge': len(sub) / sub[['sender_user','receiver_user']].drop_duplicates().shape[0],
        'time_start': sub['txn_time'].min(),
        'time_end': sub['txn_time'].max(),
        'amount_mean': sub['amount'].mean(),
        'amount_median': sub['amount'].median(),
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

# Отдельно показываем, что все мошеннические транзакции завершены успешно
display(pd.crosstab(fraud['label'], fraud['status']))

,label,rows,unique_senders,unique_receivers,unique_edges,avg_tx_per_sender,avg_tx_per_edge,time_start,time_end,amount_mean,amount_median
0,F_bot,721,39,4,155,18.487179,4.651613,2011-06-03 02:44:48,2011-10-01 00:19:53,10689.557268,9784.17
1,F-Mule-With,717,4,6,24,179.250000,29.875000,2011-06-03 17:58:29,2011-09-30 21:53:41,10591.167001,9686.33
2,F_SevWith,240,60,6,209,4.000000,1.148325,2011-06-05 22:24:34,2011-06-15 01:56:17,13007.547417,10808.89


status,SU
label,
F-Mule-With,717
F_SevWith,240
F_bot,721


In [5]:
# Вспомогательные функции для графа
def weighted_bipartite_figure(sub, title, left_label='Отправители', right_label='Получатели'):
    edges = (
        sub.groupby(['sender_user', 'receiver_user'])
           .size()
           .reset_index(name='weight')
           .sort_values('weight', ascending=False)
    )

    senders = edges['sender_user'].unique().tolist()
    receivers = edges['receiver_user'].unique().tolist()

    G = nx.DiGraph()
    for _, r in edges.iterrows():
        G.add_edge(r['sender_user'], r['receiver_user'], weight=int(r['weight']))

    # Раскладка: отправители слева, получатели справа
    y_left = np.linspace(1, 0, len(senders)) if len(senders) > 1 else np.array([0.5])
    y_right = np.linspace(1, 0, len(receivers)) if len(receivers) > 1 else np.array([0.5])
    pos = {}
    for n, y in zip(senders, y_left):
        pos[n] = (0, float(y))
    for n, y in zip(receivers, y_right):
        pos[n] = (1, float(y))

    # Рёбра
    edge_x, edge_y = [], []
    for u, v, data in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.8, color='rgba(120,120,120,0.35)'),
        hoverinfo='none',
        mode='lines',
        showlegend=False
    )

    sender_counts = sub['sender_user'].value_counts()
    receiver_counts = sub['receiver_user'].value_counts()
    sender_unique_rx = sub.groupby('sender_user')['receiver_user'].nunique()
    receiver_unique_tx = sub.groupby('receiver_user')['sender_user'].nunique()

    sender_x = [pos[n][0] for n in senders]
    sender_y = [pos[n][1] for n in senders]
    sender_size = [8 + 2.2 * sender_counts[n] for n in senders]
    sender_text = [
        f'{n}<br>Транзакций: {sender_counts[n]}<br>Уникальных получателей: {sender_unique_rx[n]}'
        for n in senders
    ]

    receiver_x = [pos[n][0] for n in receivers]
    receiver_y = [pos[n][1] for n in receivers]
    receiver_size = [12 + 2.2 * receiver_counts[n] for n in receivers]
    receiver_text = [
        f'{n}<br>Транзакций: {receiver_counts[n]}<br>Уникальных отправителей: {receiver_unique_tx[n]}'
        for n in receivers
    ]

    sender_trace = go.Scatter(
        x=sender_x, y=sender_y,
        mode='markers+text',
        text=senders,
        textposition='middle left',
        hovertext=sender_text,
        hoverinfo='text',
        marker=dict(size=sender_size, color='royalblue', line=dict(width=1, color='white')),
        name=left_label
    )

    receiver_trace = go.Scatter(
        x=receiver_x, y=receiver_y,
        mode='markers+text',
        text=receivers,
        textposition='middle right',
        hovertext=receiver_text,
        hoverinfo='text',
        marker=dict(size=receiver_size, color='firebrick', line=dict(width=1, color='white')),
        name=right_label
    )

    fig = go.Figure(data=[edge_trace, sender_trace, receiver_trace])
    fig.update_layout(
        title=title,
        title_x=0.5,
        template='plotly_white',
        width=1150,
        height=max(600, 18 * max(len(senders), len(receivers))),
        margin=dict(l=20, r=20, t=60, b=20),
        xaxis=dict(visible=False, range=[-0.2, 1.2]),
        yaxis=dict(visible=False),
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    )
    return fig, edges

def daily_counts_figure(sub, title):
    daily = sub.groupby(sub['txn_time'].dt.date).size().reset_index(name='count')
    daily.columns = ['date', 'count']
    fig = px.line(daily, x='date', y='count', markers=True, title=title)
    fig.update_layout(template='plotly_white', width=1000, height=350, title_x=0.5)
    return fig, daily

def amount_figure(sub, title):
    fig = px.box(
        sub,
        x='label',
        y='amount',
        points='outliers',
        title=title
    )
    fig.update_layout(template='plotly_white', width=1000, height=420, title_x=0.5, xaxis_title='')
    return fig

## 2. Схема №1 — botnet-паттерн (`F_bot`)

Признаки, которые проверяем:

- множество отправителей, но очень маленькое число получателей;
- почти полный двудольный граф;
- повторяющиеся переводы между теми же узлами;
- одинаковая успешность операций;
- активность укладывается в ограниченный временной интервал.

In [6]:


bot = fraud[fraud['label'] == 'F_bot'].copy()

bot_stats = pd.DataFrame([{
    'rows': len(bot),
    'unique_senders': bot['sender_user'].nunique(),
    'unique_receivers': bot['receiver_user'].nunique(),
    'unique_edges': bot[['sender_user','receiver_user']].drop_duplicates().shape[0],
    'possible_edges': bot['sender_user'].nunique() * bot['receiver_user'].nunique(),
    'avg_tx_per_sender': len(bot) / bot['sender_user'].nunique(),
    'avg_tx_per_edge': len(bot) / bot[['sender_user','receiver_user']].drop_duplicates().shape[0],
    'time_start': bot['txn_time'].min(),
    'time_end': bot['txn_time'].max(),
    'amount_mean': bot['amount'].mean(),
    'amount_median': bot['amount'].median(),
}])

display(bot_stats)

fig_bot_graph, bot_edges = weighted_bipartite_figure(
    bot,
    title='Граф мошеннической схемы F_bot: почти полный двудольный граф отправителей и получателей'
)
fig_bot_graph.show()

fig_bot_time, bot_daily = daily_counts_figure(bot, 'Дневная динамика транзакций F_bot')
fig_bot_time.show()

fig_bot_amount = amount_figure(bot, 'Распределение сумм транзакций в схеме F_bot')
fig_bot_amount.show()

# Дополнительная проверка структуры: сколько уникальных получателей у каждого отправителя
sender_receiver_counts = bot.groupby('sender_user')['receiver_user'].nunique().value_counts().sort_index()
display(sender_receiver_counts.to_frame('число отправителей'))

,rows,unique_senders,unique_receivers,unique_edges,possible_edges,avg_tx_per_sender,avg_tx_per_edge,time_start,time_end,amount_mean,amount_median
0,721,39,4,155,156,18.487179,4.651613,2011-06-03 02:44:48,2011-10-01 00:19:53,10689.557268,9784.17


,число отправителей
receiver_user,
3,1
4,38


**Вывод по схеме botnet.**   
Граф показывает почти полный двудольный шаблон: отправители соединены практически со всеми 4 получателями, а каждый получатель принимает переводы от большинства отправителей. Такой рисунок соответствует координированной атаке, где множество скомпрометированных кошельков переводят деньги в небольшой набор «сборщиков» средств.

## 3. Схема №2 — вывод средств через агентов (`F-Mule-With` + `F_SevWith`)

Эта часть данных описывает каскадный cash-out-паттерн:

- деньги выводятся через небольшую сеть агентских аккаунтов `RET`;
- есть два масштаба поведения:
  - `F-Mule-With` — очень плотное ядро, где 4 отправителя многократно взаимодействуют с 6 агентами;
  - `F_SevWith` — более широкая волна: 60 отправителей и те же 6 агентов, но у каждого отправителя только 3–4 получателя.
- один и тот же набор агентских узлов появляется в обеих подвыборках, что указывает на общую инфраструктуру обналичивания.

In [7]:
cash = fraud[fraud['label'].isin(['F-Mule-With', 'F_SevWith'])].copy()

cash_stats = pd.DataFrame([{
    'rows': len(cash),
    'unique_senders': cash['sender_user'].nunique(),
    'unique_receivers': cash['receiver_user'].nunique(),
    'unique_edges': cash[['sender_user','receiver_user']].drop_duplicates().shape[0],
    'possible_edges': cash['sender_user'].nunique() * cash['receiver_user'].nunique(),
    'avg_tx_per_sender': len(cash) / cash['sender_user'].nunique(),
    'avg_tx_per_edge': len(cash) / cash[['sender_user','receiver_user']].drop_duplicates().shape[0],
    'time_start': cash['txn_time'].min(),
    'time_end': cash['txn_time'].max(),
    'amount_mean': cash['amount'].mean(),
    'amount_median': cash['amount'].median(),
}])

display(cash_stats)
display(pd.crosstab(cash['label'], cash['receiver_type']))

fig_cash_graph, cash_edges = weighted_bipartite_figure(
    cash,
    title='Граф cash-out схемы: вывод средств через ограниченную сеть агентских узлов'
)
fig_cash_graph.show()

fig_cash_time, cash_daily = daily_counts_figure(cash, 'Дневная динамика транзакций cash-out схемы')
fig_cash_time.show()

fig_cash_amount = amount_figure(cash, 'Распределение сумм транзакций в cash-out схеме')
fig_cash_amount.show()

# Раздельная детализация двух подтипов
sub_stats = []
for label in ['F-Mule-With', 'F_SevWith']:
    sub = cash[cash['label'] == label]
    sub_stats.append({
        'label': label,
        'rows': len(sub),
        'senders': sub['sender_user'].nunique(),
        'receivers': sub['receiver_user'].nunique(),
        'unique_edges': sub[['sender_user','receiver_user']].drop_duplicates().shape[0],
        'avg_receivers_per_sender': sub.groupby('sender_user')['receiver_user'].nunique().mean(),
        'avg_senders_per_receiver': sub.groupby('receiver_user')['sender_user'].nunique().mean(),
    })
display(pd.DataFrame(sub_stats))

,rows,unique_senders,unique_receivers,unique_edges,possible_edges,avg_tx_per_sender,avg_tx_per_edge,time_start,time_end,amount_mean,amount_median
0,957,63,6,229,378,15.190476,4.179039,2011-06-03 17:58:29,2011-09-30 21:53:41,11197.15582,9975.46


receiver_type,RET
label,
F-Mule-With,717
F_SevWith,240


,label,rows,senders,receivers,unique_edges,avg_receivers_per_sender,avg_senders_per_receiver
0,F-Mule-With,717,4,6,24,6.000000,4.000000
1,F_SevWith,240,60,6,209,3.483333,34.833333


**Вывод по cash-out схеме.**  
Здесь виден другой паттерн: не широкий веер между множеством обычных пользователей, а опора на фиксированную агентскую инфраструктуру.  
`F-Mule-With` выглядит как плотное ядро из нескольких активных отправителей, а `F_SevWith` — как более массовая волна, которая всё равно стягивается в одни и те же 6 агентских узлов. Это похоже на этап обналичивания после компрометации аккаунтов или устройства.

## 4. Итоговые признаки мошенничества

По результатам графового анализа можно сформулировать такие признаки:

1. **Сильная концентрация связей**: небольшое число получателей аккумулирует большой поток транзакций.
2. **Почти полные двудольные подграфы**: мошеннические узлы связаны почти со всеми узлами другой доли.
3. **Повторяемость маршрутов**: одни и те же отправители и получатели встречаются много раз.
4. **Стабильные агентские узлы** в cash-out-схеме: вывод средств идёт через ограниченный набор `RET`-аккаунтов.
5. **Отсутствие ошибок статуса**: все мошеннические транзакции завершены успешно, что делает сеть ещё более подозрительной.

**1. Botnet-схема (`F_bot`).**  
Множество скомпрометированных пользовательских аккаунтов выполняют переводы на очень маленький набор получателей. Граф почти полностью двудольный, а получатели собирают деньги от большинства отправителей.

**2. Cash-out / вывод средств через агентов (`F-Mule-With` + `F_SevWith`).**  
Средства выводятся через фиксированную сеть из 6 агентских узлов. Внутри этой схемы видно два масштаба: плотное ядро из 4 активных отправителей и более массовая волна из 60 отправителей, но оба подтипа стягиваются к одной и той же инфраструктуре обналичивания.